# Sherdog ufc web crawler and analysis

In [4]:
import pandas as pd

fights = pd.read_csv("../data/fights.csv")
fighters = pd.read_csv("../data/fighters.csv")
events = pd.read_csv("../data/events.csv")

print(fights.shape, fighters.shape, events.shape)


(8989, 23) (3256, 12) (836, 7)


In [10]:
# move fights from wide to long format
# this will allow us to do some nice chronological trend analysis
a = fights.rename(columns={"fighter_a_id": "fighter_id", "fighter_a_name": "fighter_name",
                           "fighter_b_id": "opponent_id", "fighter_b_name": "opponent_name"})
b = fights.rename(columns={"fighter_b_id": "fighter_id", "fighter_b_name": "fighter_name",
                           "fighter_a_id": "opponent_id", "fighter_a_name": "opponent_name"})
long_df = pd.concat([a, b], ignore_index=True)

print(long_df.shape)

long_df[long_df["fighter_id"] == 50529].sort_values("event_date")[
    ["fighter_id", "event_date", "fighter_name", "opponent_name", 'winner_id']
]

(17978, 23)


,fighter_id,event_date,fighter_name,opponent_name,winner_id
6053,50529,2011-01-01,Dustin Poirier,Josh Grispi,50529.0
5942,50529,2011-06-11,Dustin Poirier,Jason Young,50529.0
7967,50529,2011-11-12,Dustin Poirier,Pablo Garza,50529.0
7885,50529,2012-02-04,Dustin Poirier,Max Holloway,50529.0
16801,50529,2012-05-15,Dustin Poirier,Chan Sung Jung,36155.0
7595,50529,2012-12-15,Dustin Poirier,Jonathan Brookins,50529.0
16525,50529,2013-02-16,Dustin Poirier,Cub Swanson,11002.0
7336,50529,2013-08-31,Dustin Poirier,Erik Koch,50529.0
7198,50529,2013-12-28,Dustin Poirier,Diego Brandao,50529.0
7068,50529,2014-04-16,Dustin Poirier,Akira Corassani,50529.0


In [12]:
# check for no contests
# right now if there is no context the winner id is NaN
def get_result(row):
    if row["outcome_type"] != "win":
        return row["outcome_type"]  # draw / nc / unknown
    
    return "win" if row["winner_id"] == row["fighter_id"] else "loss"

long_df["result"] = long_df.apply(get_result, axis=1)

long_df[long_df["fighter_id"] == 50529].sort_values("event_date")[
    ["event_date", "opponent_name", "result"]
]


,event_date,opponent_name,result
6053,2011-01-01,Josh Grispi,win
5942,2011-06-11,Jason Young,win
7967,2011-11-12,Pablo Garza,win
7885,2012-02-04,Max Holloway,win
16801,2012-05-15,Chan Sung Jung,loss
7595,2012-12-15,Jonathan Brookins,win
16525,2013-02-16,Cub Swanson,loss
7336,2013-08-31,Erik Koch,win
7198,2013-12-28,Diego Brandao,win
7068,2014-04-16,Akira Corassani,win


In [15]:
# leakage safe days since last fight
long_df["event_date"] = pd.to_datetime(long_df["event_date"]) # turn the string date into panda datetime
long_df = long_df.sort_values(["fighter_id", "event_date"]).reset_index(drop=True) # srot by the fighter id and event date

# clever part. diff calc rows val minus the previous row.
long_df["days_since_prior"] = long_df.groupby("fighter_id")["event_date"].diff().dt.days

long_df[long_df["fighter_id"] == 50529][["event_date", "opponent_name", "days_since_prior"]].head(6)


,event_date,opponent_name,days_since_prior
8464,2011-01-01,Josh Grispi,NaN
8465,2011-06-11,Jason Young,161.0
8466,2011-11-12,Pablo Garza,154.0
8467,2012-02-04,Max Holloway,84.0
8468,2012-05-15,Chan Sung Jung,101.0
8469,2012-12-15,Jonathan Brookins,214.0


In [18]:
# a fighters win rate computed only from previous fights
long_df["is_win"] = (long_df["result"] == "win").astype(int)

long_df["win_rate_entering"] = (long_df.groupby("fighter_id")["is_win"]
                       .apply(lambda s: s.shift().expanding().mean())
                       .reset_index(drop=True))

long_df[long_df["fighter_id"] == 50529][["event_date", "opponent_name", "result", "win_rate_entering"]].head(8)


,event_date,opponent_name,result,win_rate_entering
8464,2011-01-01,Josh Grispi,win,NaN
8465,2011-06-11,Jason Young,win,1.000000
8466,2011-11-12,Pablo Garza,win,1.000000
8467,2012-02-04,Max Holloway,win,1.000000
8468,2012-05-15,Chan Sung Jung,loss,1.000000
8469,2012-12-15,Jonathan Brookins,win,0.800000
8470,2013-02-16,Cub Swanson,loss,0.833333
8471,2013-08-31,Erik Koch,win,0.714286


In [19]:
# we would also like the age of a fighter at the time of one of his fights
fighters["birth_date"] = pd.to_datetime(fighters["birth_date"])

# keep every row of df, attatch birth date where a matching fighter id exists in fighters
long_df = long_df.merge(fighters[["fighter_id", "birth_date"]], on="fighter_id", how="left")

long_df["age_at_fight"] = (long_df["event_date"] - long_df["birth_date"]).dt.days / 365.25

long_df[long_df["fighter_id"] == 50529][["event_date", "opponent_name", "age_at_fight"]].head(4)

,event_date,opponent_name,age_at_fight
8464,2011-01-01,Josh Grispi,21.949350
8465,2011-06-11,Jason Young,22.390144
8466,2011-11-12,Pablo Garza,22.811773
8467,2012-02-04,Max Holloway,23.041752


In [20]:
FINISH_METHODS = {"KO", "TKO", "Submission", "Technical Submission"}
long_df["is_finish"] = long_df["method_category"].isin(FINISH_METHODS).astype(int)

# group by fighter id
# for s (every fighters results 1, 0, 0, 1 etc) apply lambda function
# move each val one step back
# builds cum window from start fighter history to this point
# mean averages all fights. on avg how many of this fighters fight are a finish
long_df["finish_rate_entering"] = (
    long_df.groupby("fighter_id")["is_finish"]
    .apply(lambda s: s.shift().expanding().mean())
    .reset_index(drop=True)
)

long_df[long_df["fighter_id"] == 50529][
    ["event_date", "opponent_name", "method_category", "is_finish", "finish_rate_entering"]
].head(6)

,event_date,opponent_name,method_category,is_finish,finish_rate_entering
8464,2011-01-01,Josh Grispi,Decision,0,NaN
8465,2011-06-11,Jason Young,Decision,0,0.000000
8466,2011-11-12,Pablo Garza,Submission,1,0.000000
8467,2012-02-04,Max Holloway,Submission,1,0.333333
8468,2012-05-15,Chan Sung Jung,Technical Submission,1,0.500000
8469,2012-12-15,Jonathan Brookins,Submission,1,0.600000


In [21]:
# merge the entering features we calculated into the wide fights.csv
# for each fight we want to be able to see both fighters pre fight stats
features = long_df[[
    "fight_id", "fighter_id", "win_rate_entering", "finish_rate_entering",
    "days_since_prior", "age_at_fight",
]]

a_features = features.rename(columns={
    "fighter_id": "fighter_a_id",
    "win_rate_entering": "a_win_rate_entering",
    "finish_rate_entering": "a_finish_rate_entering",
    "days_since_prior": "a_days_since_prior",
    "age_at_fight": "a_age_at_fight",
})
b_features = features.rename(columns={
    "fighter_id": "fighter_b_id",
    "win_rate_entering": "b_win_rate_entering",
    "finish_rate_entering": "b_finish_rate_entering",
    "days_since_prior": "b_days_since_prior",
    "age_at_fight": "b_age_at_fight",
})

fights_model = fights.merge(a_features, on=["fight_id", "fighter_a_id"], how="left")
fights_model = fights_model.merge(b_features, on=["fight_id", "fighter_b_id"], how="left")

fights_model[fights_model["fight_id"] == "101617-12"][
    ["event_date", "fighter_a_name", "fighter_b_name", "b_win_rate_entering", "b_finish_rate_entering"]
]

,event_date,fighter_a_name,fighter_b_name,b_win_rate_entering,b_finish_rate_entering
2033,2024-06-01,Islam Makhachev,Dustin Poirier,0.733333,0.7
